# Does a properly-trained memory still have a short horizon?

**The gap this closes.** The horizon result -- forget gate `alpha ~ 0.74`, so time constant
`tau = -1/ln(1-alpha) ~ 0.74 chunks ~ 6 positions` against a 256-position context -- was
measured on `dim=64`, 256 memory units, 3000 steps. That is a toy. A reviewer will ask
whether a real model learns the same thing, and right now we cannot answer.

This notebook scales the model and re-measures. One variable moves: **width**.

| | current claim | this run |
|---|---|---|
| transformer dim | 64 | **384** |
| memory units | 256 | **1536** |
| training steps | 3,000 | **25,000** |

Everything else -- depth, chunk size, context, LR, the measurement code -- is unchanged, so
the numbers are directly comparable.

### The second thing this settles

`alpha` was still **moving** in the original runs: 0.579 at 1.2k steps, 0.741 at 3k. So
0.74 might be a converged value, or it might just be where training stopped. This notebook
measures `alpha` at checkpoints throughout, which tells you which.

**Three outcomes, all publishable:**
- `alpha` converges near 0.74 -> the short horizon is a property of the trained memory, and
  your headline becomes a real claim rather than a toy observation.
- `alpha` keeps climbing -> the memory forgets *faster* the more you train it. Stronger, and
  more surprising.
- `alpha` falls at scale -> the short horizon was a small-model artifact. Report it; the
  structural claim (one scalar = one timescale) survives regardless.

Runtime: ~1-2 h on an A100 at 25k steps. Drop `STEPS` to 10k for a faster first answer.

## 1 - Setup

In [ ]:
!pip install -q titans-pytorch
!git clone -q https://github.com/thebnbrkr/marv-titan.git /content/marv-titan 2>/dev/null || true
!git clone -q --depth 1 https://github.com/lucidrains/titans-pytorch.git /content/titans-src 2>/dev/null || true
import sys; sys.path.insert(0, '/content/marv-titan/experiments')

import torch, numpy as np, time, json, gzip, os
from titans_pytorch import MemoryAsContextTransformer, MemoryMLP
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")

In [ ]:
# the SAME measurement code that produced the dim-64 numbers
from titans_real_text import (load_enwik8, sample_batch, _cos,
                              inspect_decay_gate, consecutive_write_alignment)

DATA = "/content/titans-src/data/enwik8.gz"
assert os.path.exists(DATA), "enwik8 not found -- re-run the clone cell"
data_train, data_val = load_enwik8(DATA)
print(f"enwik8: train {len(data_train):,} val {len(data_val):,} bytes")

## 2 - The scaled model

`NeuralMemory` asserts `dim_head == dim` when `heads == 1` (neural_memory.py:304), so the
memory's width follows the transformer's. At `dim=384` the memory MLP becomes
`384 -> 1536 -> 384`: **1536 units instead of 256**.

Attention head config is scaled conventionally (`dim_head=64, heads=6`). Depth, chunk size,
context, and LR are held at the original values so only width changes.

In [ ]:
DIM        = 384      # was 64.  try 256 if 384 is too slow
DEPTH      = 4        # unchanged
SEG        = 8        # neural_memory_segment_len -- unchanged, so tau is in the same units
SEQ_LEN    = 256      # unchanged
BATCH      = 8        # unchanged
LR         = 2e-4     # unchanged
STEPS      = 25_000   # was 3,000
CHECKPOINTS = [1000, 3000, 6000, 12000, 25000]   # measure alpha at each
PASSAGE_LEN = 1024

def build_scaled(dim=DIM):
    return MemoryAsContextTransformer(
        num_tokens=256, dim=dim, depth=DEPTH,
        segment_len=32, neural_memory_segment_len=SEG,
        num_persist_mem_tokens=4, num_longterm_mem_tokens=4,
        neural_memory_layers=(2,),
        dim_head=64, heads=6,                                   # scaled attention heads
        neural_memory_model=MemoryMLP(dim, depth=2, expansion_factor=4.),
        neural_memory_kwargs=dict(dim_head=dim, heads=1),       # heads=1 => dim_head must == dim
        use_flex_attn=False)

m = build_scaled().to(DEVICE)
n = sum(p.numel() for p in m.parameters())
mem_units = int(DIM*4)
print(f"dim {DIM} | memory {DIM} -> {mem_units} -> {DIM}  ({mem_units} units, was 256)")
print(f"total params: {n/1e6:.1f}M")
del m; torch.cuda.empty_cache()

## 3 - Measurement helpers

`measure_alpha` hooks the same `to_decay_factor` module that `inspect_decay_gate` hooks, but
returns the number instead of printing it, so we can call it at every checkpoint.

In [ ]:
@torch.no_grad()
def measure_alpha(model, passage):
    """Mean learned forget gate on a real forward pass. Same hook as inspect_decay_gate."""
    mem = next(g[4] for g in model.layers if g[4] is not None)
    got = {}
    h = mem.to_decay_factor.register_forward_hook(
        lambda mo, i, o: got.__setitem__("a", o.sigmoid().detach()))
    model(passage.unsqueeze(0).to(DEVICE), return_cache=True)
    h.remove()
    return float(got["a"].mean())

def tau_of(alpha):
    """Memory horizon in chunks, then positions."""
    t = -1.0/np.log(1.0-alpha)
    return t, t*SEG

@torch.no_grad()
def val_loss(model, n_batches=20):
    model.eval()
    v = float(np.mean([model(sample_batch(data_val, SEQ_LEN, BATCH).to(DEVICE),
                             return_loss=True).item() for _ in range(n_batches)]))
    model.train(); return v

@torch.no_grad()
def norm_ratio(model, passage):
    model.eval()
    _, cache = model(passage.unsqueeze(0).to(DEVICE), return_cache=True)
    U0 = cache[2][0].updates["model.weights.0"].detach()[0].cpu().numpy()
    g_in, g_out = U0[1], U0[-1]
    gc = _cos(g_in, g_out, axis=0)
    nr = (np.linalg.norm(g_out,axis=0)+1e-9)/(np.linalg.norm(g_in,axis=0)+1e-9)
    moved = gc < 0.99
    model.train()
    return float(nr[moved].mean()) if moved.any() else float("nan"), int(moved.sum()), len(gc)

## 4 - Train, measuring alpha as it goes

One optimizer for the whole run (constructing a fresh Adam at each checkpoint would reset
the moment estimates and change the trajectory -- a bug we hit earlier).

In [ ]:
torch.manual_seed(0); np.random.seed(0)
model = build_scaled().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)     # ONE optimizer for the whole run
passage = data_val[:PASSAGE_LEN]

rows, done, t0 = [], 0, time.time()
model.train()
for target in CHECKPOINTS:
    for step in range(target - done):
        loss = model(sample_batch(data_train, SEQ_LEN, BATCH).to(DEVICE), return_loss=True)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step()
    done = target

    a = measure_alpha(model, passage)
    tc, tp = tau_of(a)
    nr, moved, tot = norm_ratio(model, passage)
    v = val_loss(model)
    rows.append({"steps": target, "alpha": a, "tau_chunks": tc, "tau_positions": tp,
                 "val_loss": v, "norm_ratio": nr, "units_moved": moved, "n_units": tot,
                 "minutes": (time.time()-t0)/60})
    print(f"[{target:>6} steps] val {v:.3f} | alpha {a:.4f} | tau {tc:.2f} chunks "
          f"= {tp:.1f} positions | norm_ratio {nr:.2f} | {(time.time()-t0)/60:.0f} min", flush=True)
    json.dump(rows, open("titans_scaled.json","w"), indent=2)

torch.save(model.state_dict(), f"titans_dim{DIM}_{STEPS}.pt")
print("\ndone -- checkpoint saved")

## 5 - Did alpha converge, and what is the horizon?

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.DataFrame(rows); display(df.round(4))

REF = {"alpha": 0.741, "tau_positions": 5.9, "val_loss": 1.974}   # dim-64, 3k steps
last = df.iloc[-1]
print(f"\ndim-64  baseline : alpha {REF['alpha']:.3f} | tau {REF['tau_positions']:.1f} positions | val {REF['val_loss']:.3f}")
print(f"dim-{DIM} this run : alpha {last.alpha:.3f} | tau {last.tau_positions:.1f} positions | val {last.val_loss:.3f}")
print(f"\ncontext is {SEQ_LEN} positions -> memory horizon is {last.tau_positions/SEQ_LEN*100:.1f}% of context")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for a in ax:
    a.set_facecolor("#fcfcfb"); a.grid(True, color="#e8e8e3", lw=.8); a.set_axisbelow(True)
    for s in ("top","right"): a.spines[s].set_visible(False)
    for s in ("left","bottom"): a.spines[s].set_color("#d4d4cd")
    a.tick_params(colors="#6b6b64", labelsize=9)
fig.patch.set_facecolor("#fcfcfb")

ax[0].plot(df.steps, df.alpha, marker="o", ms=7, lw=2.2, color="#2a78d6",
           markeredgecolor="#fcfcfb", markeredgewidth=2)
ax[0].axhline(REF["alpha"], color="#eb6834", lw=1.5, ls="--")
ax[0].annotate("dim-64 value (0.741)", (df.steps.iloc[0], REF["alpha"]),
               textcoords="offset points", xytext=(4,6), fontsize=9, color="#eb6834")
ax[0].set_xlabel("training steps", fontsize=10, color="#54544c")
ax[0].set_ylabel("learned forget gate  alpha", fontsize=10, color="#54544c")
ax[0].set_title("Does the gate converge?", fontsize=11.5, color="#33332f", loc="left", pad=10)

ax[1].plot(df.steps, df.tau_positions, marker="o", ms=7, lw=2.2, color="#184f95",
           markeredgecolor="#fcfcfb", markeredgewidth=2)
ax[1].axhline(SEQ_LEN, color="#9a9a93", lw=1, ls="--")
ax[1].annotate(f"full context ({SEQ_LEN})", (df.steps.iloc[0], SEQ_LEN),
               textcoords="offset points", xytext=(4,-13), fontsize=9, color="#6b6b64")
ax[1].set_yscale("log")
ax[1].set_xlabel("training steps", fontsize=10, color="#54544c")
ax[1].set_ylabel("memory horizon tau (positions, log)", fontsize=10, color="#54544c")
ax[1].set_title("How far back can the memory see?", fontsize=11.5, color="#33332f", loc="left", pad=10)
plt.tight_layout(); plt.savefig("titans_scaled.png", dpi=190); plt.show()

## How to read this

**Look at the alpha curve first.** If it flattens, 0.74 was a converged property and the
horizon claim is real. If it is still climbing at 25k steps, say so -- "the gate had not
converged at 25k steps; the horizon is an upper bound on retention" is honest and still
makes the point.

**Then compare to the dim-64 baseline.** The memory here has 1536 units instead of 256 and
the model is ~6x wider. If tau stays a few positions, scale does not rescue the horizon --
that is the result that turns a toy observation into a claim about the architecture.

**Sanity check the val loss.** It should be clearly below the dim-64 run's 1.974. If it is
not, the larger model has not trained properly and nothing else here is trustworthy.

**What to write.** Replace Section 3's sentence with the larger model's number and report
the dim-64 value alongside, so the reader sees the horizon measured at two scales. Keep the
bound that this is the memory's decay, not the model's total context reach.